In [1]:
# Define parameters
keyword = '/g/11c265kd70'
gprop = 'youtube'
geo = 'IN'
timeframe = '2025-06-15 2025-12-29'
params = {
    "api_key" : "b9470e54fd35b3f2174c686b526660af023bbcb1521aa1211194bef5a5a663c4",
    "engine": "google_trends",
    "q": keyword,
    "geo": geo,
    "date": timeframe,
    "data_type": "TIMESERIES",
    "tz": "330",
    "gprop": gprop if gprop != "web" else ""
}

In [2]:
from gtrends.chunk.fetch import SerpApiTrendsFetcher
fetcher = SerpApiTrendsFetcher(geo="IN")
fetcher.initialize()

Successfully connected to the database
Executing query: 
            SELECT `id`, `token`
            FROM `external_tokens`
            WHERE `valid_from` < CURRENT_TIMESTAMP
            AND `platform` = 'serpapi'
            AND `token_type` = 'api_key'
        
Database connection closed.


In [3]:
data = fetcher.fetch_chunk(
    keyword="/g/11c265kd70",
    start="2025-06-01",
    end="2025-12-31",
    gprop="youtube"
)

Successfully connected to the database
Executing query: 
            UPDATE external_tokens
            SET last_used_at = CURRENT_TIMESTAMP
            WHERE token = 'b9470e54fd35b3f2174c686b526660af023bbcb1521aa1211194bef5a5a663c4'
            
Database connection closed.


In [38]:
from datetime import datetime, timedelta
from statistics import mean
import gtrends.db.utils as db_utils
from gtrends.db.keywords import GoogleTrendsRepository

class GoogleTrendSyncer:
    """
    Handles normalization and continuity of Google Trends time series
    across overlapping fetch windows.
     - Feeds it into the database while also updating the past entries
    """

    DATE_FMT = "%d-%m-%Y"

    def __init__(self):
        self.gtr = GoogleTrendsRepository()
    
    def _find_overlap(self, past, new):
        past_map = {p["date"]: p["value"] for p in past}
        overlap = []

        for row in new:
            if row["date"] in past_map:
                overlap.append((past_map[row["date"]], row["value"]))

        return overlap
    
    def compute_scaling_factor(self, overlap):
        """
        Scaling factor = mean(past_values) / mean(new_values)
        """
        if len(overlap) < 1:
            return 1.0  # insufficient overlap → no scaling

        past_vals = [p for p, _ in overlap]
        new_vals = [n for _, n in overlap]

        if mean(new_vals) == 0:
            return 1.0

        return mean(past_vals) / mean(new_vals)
    
    def normalize_new_data(self, new_data, scale):
        normalized = []

        for row in new_data:
            normalized.append({
                "date": row["date"],
                "value": round(row["value"] * scale, 2)
            })

        return normalized
    
    def merge_timeseries(self, past, normalized_new):
        merged = {row["date"]: row["value"] for row in past}

        for row in normalized_new:
            merged[row["date"]] = row["value"]

        return [
            {"date": d, "value": v}
            for d, v in sorted(merged.items(), key=lambda x: datetime.strptime(x[0], self.DATE_FMT))
        ]
    
    def upsert_timeseries(self, keyword, geo, data):
        query = """
        INSERT INTO google_trends_timeseries (keyword, geo, date, value)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE value = VALUES(value)
        """

        params = [
            (
                keyword,
                geo,
                datetime.strptime(row["date"], self.DATE_FMT),
                row["value"]
            )
            for row in data
        ]

        self.db.executemany(query, params)
        self.db.commit()

    def generate_windows(self, start, end, window_days = 240, overlap_days = 30):
        w_end = end

        windows = []
        while w_end > start:
            w_start = max(w_end - timedelta(days = window_days), start) # enforces 30 days overlap
            windows.append((w_start.isoformat(), w_end.isoformat()))
            w_end = w_end - timedelta(days = window_days - overlap_days)  # shift left by 210 days

        return windows[::-1]

In [ ]:
gts = GoogleTrendSyncer()

def sync(search_id):
    """
    fetch_fn(start, end) → returns list of {date, value}
    """
    def get_windows(search_id):
        past = gts.gtr.get_past_data(search_id)
        if past:
            start = datetime.fromtimestamp(past[-1].get('timestamp')) - timedelta(days=30)
        else:
            start = datetime.now().date() - timedelta(days = 4 * 365)
        end = datetime.now().date()

        windows = gts.generate_windows(start, end)
        return windows
    
    windows = get_windows(search_id)
    stitched = []

    for start, end in windows:
        chunk = fetch_fn(start, end)

        if not stitched:
            stitched = chunk
            continue

        overlap = self._find_overlap(stitched, chunk)
        scale = self.compute_scaling_factor(overlap)
        chunk = self.normalize_new_data(chunk, scale)
        stitched = self.merge_timeseries(stitched, chunk)

    self.upsert_timeseries(keyword, geo, stitched)

In [ ]:
from gtrends.chunk.fetch import SerpApiTrendsFetcher
stf = SerpApiTrendsFetcher()
stf.initialize()

gtr = GoogleTrendsRepository()
searches_pending = gtr.searches_to_update()

current_search = searches_pending[0]
windows = sync(current_search.get('id'))

start, end = windows[0]
chunk = stf.fetch_chunk(keyword = current_search.get('search_keyword')
                        start = start, end)

In [ ]:
from gtrends.db.keywords import GoogleTrendsRepository

gtr = GoogleTrendsRepository()
past = gtr.get_past_data(1)

if not past:
    

# overlap = gts._find_overlap(past, new_data)
# scale = gts.compute_scaling_factor(overlap)
# normalized = gts.normalize_new_data(new_data, scale)
# merged = gts.merge_timeseries(past, normalized)

# gts.upsert_timeseries(keyword, geo, merged)



In [6]:
past

[]